In [39]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

In [67]:
df = pd.read_csv("../data/processed/clean_housing.csv")

In [68]:
X = df.drop(columns=["PRICE"])
y = np.log1p(df["PRICE"])

In [69]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [70]:
print(X_train.shape)
print(y_train.shape)

(2133, 22)
(2133,)


In [71]:
numeric_features = [
    "LAND_AREA",
    "ROAD_ACCESS",
    "area_bedroom",
    "FLOOR",
    "area_road_interaction",
    "house_age",
    "AMENITY_COUNT",
    "bath_per_bed"
]

categorical_features = [
    "FACING",
    "LOCATION"
]

In [72]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder='passthrough'
)

In [73]:
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

In [74]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(max_iter=10000),
    "Ridge": Ridge(max_iter=10000),
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingRegressor(),
    "AdaBoost Regressor": AdaBoostRegressor(),
    "XGBRegressor": XGBRegressor(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8
    ), 
    "CatBoosting Regressor": CatBoostRegressor(verbose=False),
}

In [75]:
for name, model in models.items():
    best_pipeline = None
    best_score = -float("inf")

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    pipeline.fit(X_train, y_train)

    # Make predictions
    y_train_pred = pipeline.predict(X_train)
    y_test_pred = pipeline.predict(X_test)
    
    # Evaluate Train and Test dataset
    train_mae, train_rmse, train_r2 = evaluate_model(y_train, y_train_pred)
    test_mae, test_rmse, test_r2 = evaluate_model(y_test, y_test_pred)

    if test_r2 > best_score:
        best_score = test_r2
        best_pipeline = pipeline
    
    print(name)
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(train_mae))
    print("- R2 Score: {:.4f}".format(train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(test_mae))
    print("- R2 Score: {:.4f}".format(test_r2))
    
    print('='*35)
    print('\n')

Linear Regression
Model performance for Training set
- Root Mean Squared Error: 0.5292
- Mean Absolute Error: 0.3360
- R2 Score: 0.2609
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 0.5220
- Mean Absolute Error: 0.3500
- R2 Score: 0.2580


Lasso
Model performance for Training set
- Root Mean Squared Error: 0.6156
- Mean Absolute Error: 0.4218
- R2 Score: 0.0000
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 0.6069
- Mean Absolute Error: 0.4262
- R2 Score: -0.0031


Ridge
Model performance for Training set
- Root Mean Squared Error: 0.5295
- Mean Absolute Error: 0.3364
- R2 Score: 0.2601
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 0.5231
- Mean Absolute Error: 0.3504
- R2 Score: 0.2549


Random Forest Regressor
Model performance for Training set
- Root Mean Squared Error: 0.3626
- Mean Absolute Error: 0.1849
- R2 Score: 0.6531
--------------------

In [76]:
joblib.dump(best_pipeline, "../model/house_price_model.pkl")

['../model/house_price_model.pkl']